# 6 株価が上昇するパターンの検出

In [58]:
import os
import sys
import requests
import datetime as dt
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.rci import Rci

In [59]:
# APIの接続先（notebookコンテナでは通常 http://api:8000 が設定される）
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
# データを取得する関数
sys.path.append("/workspace/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/workspace/data'))

## 6.1 銘柄の選び方

In [60]:
get_url = f"{API_BASE_URL}/api/v1/stocks"

get_response = requests.get(get_url, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_results_sony = pd.DataFrame(results)

    print("取得件数:", len(df_results_sony))
    display(df_results_sony.head())
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/stocks
GET status: 200
取得件数: 2


,id,code,market,name,sector,memo,created_at,updated_at
0,1,7203,TSE,None,None,None,2026-04-12T01:38:35.677888Z,2026-04-12T01:38:35.677888Z
1,2,6758,TSE,None,None,None,2026-04-12T01:38:37.050763Z,2026-04-12T01:38:37.050763Z


### 6.1.2 株価チャートを確認しやすい画面の作成

In [61]:
def show_chart(
    code: str,
    name: str | None,
    market: str | None,
    start: datetime.date | str,
    end: datetime.date | str,
) -> go.Figure:
    """
    指定された銘柄コードと日付範囲に基づいて、株価チャートを生成して返す。

    Args:
        code (str): 株式の銘柄コード。
        name (str | None): 株式の銘柄名。
        market (str | None): 株式の市場。
        start (datetime.date | str): チャート表示の開始日。
        end (datetime.date | str): チャート表示の終了日。

    Returns:
        go.Figure: 描画用のPlotly Figure。
    """
    start_date = datetime.date.fromisoformat(start) if isinstance(start, str) else start
    end_date = datetime.date.fromisoformat(end) if isinstance(end, str) else end
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)

    get_url = f"{API_BASE_URL}/api/v1/stock"
    get_params = {
        "code": code,
        "market": market,
        "start": start_date.isoformat(),
        "end": end_date.isoformat(),
    }
    response = requests.get(get_url, params=get_params, timeout=60)
    response.raise_for_status()

    payload = response.json()
    rows = payload.get("results", [])
    if not rows:
        raise ValueError("指定した条件の株価データが取得できませんでした。")

    # APIレスポンス(list[dict])からDataFrameを作成
    df = pd.DataFrame(rows)

    # date列をインデックスに設定してスライス
    df = df.set_index(pd.to_datetime(df["date"])).sort_index()
    df = df[start_ts:end_ts]
    if df.empty:
        raise ValueError("指定した期間の株価データが取得できませんでした。")

    df.index = df.index.strftime("%Y-%m-%d")
    close = df["close"].values

    # 移動平均線
    df["ma5"] = ta.SMA(df["close"], timeperiod=5)
    df["ma25"] = ta.SMA(df["close"], timeperiod=25)

    # RCI
    rci = Rci()
    df["rci9"] = rci.RCI(close, timeperiod=9)
    df["rci26"] = rci.RCI(close, timeperiod=26)

    # ボリンジャーバンド±2σ
    df["upper2"], _, df["lower2"] = ta.BBANDS(
        close,
        timeperiod=25,
        nbdevup=2,
        nbdevdn=2,
        matype=ta.MA_Type.SMA,
    )

    # MACD
    macd, macd_signal, macd_hist = ta.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    df["macd"], df["macd_signal"], df["hist"] = macd, macd_signal, macd_hist

    # レイアウト設定
    layout = {
        "height": 900,
        "title": {"x": 0.5, "text": f"{name or code} {code}"},
        "xaxis": {"rangeslider": {"visible": False}, "type": "category"},
        "yaxis1": {"domain": [.36, 1.0], "title": "価格（円）", "side": "left", "tickformat": ","},
        "yaxis2": {"domain": [.30, .36]},
        "yaxis3": {"domain": [.20, .295], "title": "MACD", "side": "right"},
        "yaxis4": {"domain": [.10, .195], "title": "RCI", "side": "right"},
        "yaxis5": {"domain": [.0, .095], "title": "Volume", "s6ide": "right"},
        "plot_bgcolor": "light blue",
    }

    # データの設定
    data = [
        go.Candlestick(
            yaxis="y1",
            x=df.index,
            open=df["open"],
            high=df["high"],
            low=df["low"],
            close=df["close"],
            increasing_line_color="red",
            decreasing_line_color="gray",
            name=code,
        ),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma5"], name="MA5", line={"color": "royalblue", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma25"], name="MA25", line={"color": "lightseagreen", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["upper2"], name="", line={"color": "lavender", "width": 0}),
        go.Scatter(
            yaxis="y1",
            x=df.index,
            y=df["lower2"],
            name="BB",
            line={"color": "lavender", "width": 0},
            fill="tonexty",
            fillcolor="rgba(170,170,170,.2)",
        ),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd"], name="macd", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd_signal"], name="signal", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y3", x=df.index, y=df["hist"], name="histgram", marker={"color": "slategray"}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci9"], name="RCI9", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci26"], name="RCI26", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y5", x=df.index, y=df["volume"], name="Volume", marker=dict(color="slategray")),
    ]

    # Figure作成
    fig = go.Figure(data=data, layout=go.Layout(layout))

    # 日付表示
    fig.update_layout(
        {
            "xaxis": {
                "tickvals": df.index[::4],
                "ticktext": ["{}-{}".format(x.split("-")[1], x.split("-")[2]) for x in df.index[::4]],
            }
        }
    )

    return fig

In [62]:
# 再実行時にこのセルの過去出力を消して、フォームが埋もれないようにする
from IPython.display import clear_output
clear_output(wait=True)

# APIから銘柄リストを取得
get_url = f"{API_BASE_URL}/api/v1/stocks"
response = requests.get(get_url, timeout=60)
response.raise_for_status()

# セレクトボックス用に[(表示名, 値)]のリストを作成
stocks = response.json().get("results", [])

# VSCodeでwidgetが機能しない為コメントアウト
"""
stock_options = []
stock_map = {}
for stock in stocks:
    code = stock.get("code")
    if not code:
        continue
    market = stock.get("market") or ""
    stock_id = stock.get("id")
    value = (code, market, stock_id)
    label = f"{stock.get('name') or '銘柄名なし'} ({code}) [{market or 'N/A'}]"
    stock_options.append((label, value))
    stock_map[value] = stock

# 入力フォームを作成
stock_code_dropdown = Dropdown(
    options=stock_options,
    value=stock_options[0][1] if stock_options else None,
    description="銘柄:",
    layout=widgets.Layout(width="100%"),
    disabled=not bool(stock_options),
)

start_input = widgets.DatePicker(
    value=datetime.date(2024, 4, 1),
    description="開始日付:",
    layout=widgets.Layout(width="100%"),
)

end_input = widgets.DatePicker(
    value=datetime.date(2024, 12, 31),
    description="終了日付:",
    layout=widgets.Layout(width="100%"),
)

# 送信ボタンを作成
submit_button = Button(description="チャート表示", disabled=not bool(stock_options))
status_text = widgets.HTML(
    value="" if stock_options else "<span style='color:#c0392b'>銘柄データが取得できませんでした。APIレスポンスを確認してください。</span>"
)

# 出力エリアを作成
output = Output()

# フォーム全体を1つのコンテナにまとめて表示
form_box = widgets.VBox([
    status_text,
    stock_code_dropdown,
    start_input,
    end_input,
    submit_button,
    output,
])
display(form_box)


# ボタンが押されたときの処理
def on_submit(b: Button):
    output.clear_output(wait=True)

    with output:
        if not stock_code_dropdown.value:
            print("銘柄一覧が取得できていません。APIレスポンスを確認してください。")
            return

        selected = stock_code_dropdown.value
        stock = stock_map.get(selected, {})
        stock_code, stock_market, _ = selected
        stock_name = stock.get("name") or stock_code

        start_date = start_input.value
        end_date = end_input.value
        if start_date is None or end_date is None:
            print("開始日付と終了日付を選択してください。")
            return
        if start_date > end_date:
            print("開始日付は終了日付以前を指定してください。")
            return

        try:
            fig = show_chart(stock_code, stock_name, stock_market, start_date, end_date)
        except Exception as error:
            print(error)
            return

        fig.show()


# イベントハンドラを設定
submit_button.on_click(on_submit)
"""

'\nstock_options = []\nstock_map = {}\nfor stock in stocks:\n    code = stock.get("code")\n    if not code:\n        continue\n    market = stock.get("market") or ""\n    stock_id = stock.get("id")\n    value = (code, market, stock_id)\n    label = f"{stock.get(\'name\') or \'銘柄名なし\'} ({code}) [{market or \'N/A\'}]"\n    stock_options.append((label, value))\n    stock_map[value] = stock\n\n# 入力フォームを作成\nstock_code_dropdown = Dropdown(\n    options=stock_options,\n    value=stock_options[0][1] if stock_options else None,\n    description="銘柄:",\n    layout=widgets.Layout(width="100%"),\n    disabled=not bool(stock_options),\n)\n\nstart_input = widgets.DatePicker(\n    value=datetime.date(2024, 4, 1),\n    description="開始日付:",\n    layout=widgets.Layout(width="100%"),\n)\n\nend_input = widgets.DatePicker(\n    value=datetime.date(2024, 12, 31),\n    description="終了日付:",\n    layout=widgets.Layout(width="100%"),\n)\n\n# 送信ボタンを作成\nsubmit_button = Button(description="チャート表示", disabled=not bo

In [63]:
fig = show_chart(
    stocks[0]["code"],
    stocks[0]["name"],
    stocks[0]["market"],
    "2025-01-01",
    "2025-12-31",
)
fig.show()

HTTPError: 404 Client Error: Not Found for url: http://api:8000/api/v1/stock?code=7203&market=TSE&start=2025-01-01&end=2025-12-31

In [ ]:
# 直近2年の期間を作成
end_date = dt.date(2024, 12, 31)
start_date = end_date - dt.timedelta(days=365 * 2)

# トヨタ自動車 (Yahoo Finance: 7203.T)
name = "トヨタ自動車"
code = "7203"
market = "TSE"

get_url = f"{API_BASE_URL}/api/v1/time_series_data"
get_params = {
    "code": code,
    "market": market,
    "start": start_date.isoformat(),
    "end": end_date.isoformat(),
}

get_response = requests.get(get_url, params=get_params, timeout=60)

print("GET URL:", get_url)
print("GET status:", get_response.status_code)

if get_response.status_code == 200:
    get_json = get_response.json()
    results = get_json.get("results", [])
    df_results = pd.DataFrame(results)

    print("取得件数:", len(df_results))
    display(df_results.head())
else:
    print("GET response:", get_response.json())

GET URL: http://api:8000/api/v1/time_series_data/get
GET status: 200
取得件数: 491


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1,7203,TSE,2023-01-04,1620.543579,1625.047591,1610.184351,1619.642777,25995600,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2,7203,TSE,2023-01-05,1628.200439,1639.010069,1615.589205,1632.254051,24700200,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,3,7203,TSE,2023-01-06,1643.964478,1647.567687,1626.849231,1630.002040,22568600,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,4,7203,TSE,2023-01-10,1645.765991,1666.484446,1640.811578,1655.224416,22352300,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,5,7203,TSE,2023-01-11,1655.224487,1657.476493,1641.262049,1643.063654,19798400,1636.037387,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [64]:
df = df_results.copy()

display_start_date = dt.date(2024, 1, 1)

# date列をインデックスに設定してスライス
df = df.set_index(pd.to_datetime(df["date"])).sort_index()
df = df[display_start_date:end_date]

# レイアウト設定
layout = {
        "height": 900,
        "title": {"x": 0.5, "text": f"{name or code} {code}"},
        "xaxis": {"rangeslider": {"visible": False}, "type": "category"},
        "yaxis1": {"domain": [.36, 1.0], "title": "価格（円）", "side": "left", "tickformat": ","},
        "yaxis2": {"domain": [.30, .36]},
        "yaxis3": {"domain": [.20, .295], "title": "MACD", "side": "right"},
        "yaxis4": {"domain": [.10, .195], "title": "RCI", "side": "right"},
        "yaxis5": {"domain": [.0, .095], "title": "Volume", "side": "right"},
        "plot_bgcolor": "light blue",
}

# データの設定
data = [
        go.Candlestick(
            yaxis="y1",
            x=df.index,
            open=df["open"],
            high=df["high"],
            low=df["low"],
            close=df["close"],
            increasing_line_color="red",
            decreasing_line_color="gray",
            name=code,
        ),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma5"], name="MA5", line={"color": "royalblue", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["ma25"], name="MA25", line={"color": "lightseagreen", "width": 1.2}),
        go.Scatter(yaxis="y1", x=df.index, y=df["upper2"], name="", line={"color": "lavender", "width": 0}),
        go.Scatter(
            yaxis="y1",
            x=df.index,
            y=df["lower2"],
            name="BB",
            line={"color": "lavender", "width": 0},
            fill="tonexty",
            fillcolor="rgba(170,170,170,.2)",
        ),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd"], name="macd", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y3", x=df.index, y=df["macd_signal"], name="signal", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y3", x=df.index, y=df["hist"], name="histgram", marker={"color": "slategray"}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci9"], name="RCI9", line={"color": "magenta", "width": 1}),
        go.Scatter(yaxis="y4", x=df.index, y=df["rci26"], name="RCI26", line={"color": "green", "width": 1}),
        go.Bar(yaxis="y5", x=df.index, y=df["volume"], name="Volume", marker=dict(color="slategray")),
]

# Figure作成
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示
fig.update_layout(
        {
            "xaxis": {
                "tickvals": df.index[::4],
                "ticktext": [x.strftime("%m-%d") for x in df.index[::4]],
            }
        }
)
fig.show()

/tmp/ipykernel_29/2401714417.py:7: Pandas4Warning: Slicing with a datetime.date object is deprecated. Explicitly cast to Timestamp instead.
  df = df[display_start_date:end_date]
